In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os

PROJECT_DIR = Path("/content/drive/MyDrive/master/courses/AI_in_medicine/AI_MD_Project")

os.chdir(PROJECT_DIR)
print("Current working directory:", Path.cwd())

# Create required folders
(PROJECT_DIR / "src" / "data").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "data" / "processed").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "outputs").mkdir(parents=True, exist_ok=True)

# Make src importable as a package
(PROJECT_DIR / "src" / "__init__.py").touch()
(PROJECT_DIR / "src" / "data" / "__init__.py").touch()

**2. phq8_items**

In [ ]:
%%writefile src/data/phq8_items.py
"""
PHQ-8 item definitions.

This file connects the raw PHQ-8 label columns to the questionnaire item text
that will later be given as input to the model.
"""

PHQ8_ITEMS = {
    1: {
        "item_id": 1,
        "column": "PHQ_8NoInterest",
        "item_text": "Little interest or pleasure in doing things",
        "symptom_name": "NoInterest",
    },
    2: {
        "item_id": 2,
        "column": "PHQ_8Depressed",
        "item_text": "Feeling down, depressed, or hopeless",
        "symptom_name": "Depressed",
    },
    3: {
        "item_id": 3,
        "column": "PHQ_8Sleep",
        "item_text": "Trouble falling or staying asleep, or sleeping too much",
        "symptom_name": "Sleep",
    },
    4: {
        "item_id": 4,
        "column": "PHQ_8Tired",
        "item_text": "Feeling tired or having little energy",
        "symptom_name": "Tired",
    },
    5: {
        "item_id": 5,
        "column": "PHQ_8Appetite",
        "item_text": "Poor appetite or overeating",
        "symptom_name": "Appetite",
    },
    6: {
        "item_id": 6,
        "column": "PHQ_8Failure",
        "item_text": "Feeling bad about yourself — or that you are a failure or have let yourself or your family down",
        "symptom_name": "Failure",
    },
    7: {
        "item_id": 7,
        "column": "PHQ_8Concentrating",
        "item_text": "Trouble concentrating on things, such as reading the newspaper or watching television",
        "symptom_name": "Concentrating",
    },
    8: {
        "item_id": 8,
        "column": "PHQ_8Moving",
        "item_text": "Moving or speaking so slowly that other people could have noticed. Or the opposite — being so fidgety or restless that you have been moving around a lot more than usual",
        "symptom_name": "Moving",
    },
}


def validate_phq8_items():
    """
    Validate the static PHQ-8 mapping.
    Raises ValueError if the mapping is invalid.
    """
    required_keys = {"item_id", "column", "item_text", "symptom_name"}

    if set(PHQ8_ITEMS.keys()) != set(range(1, 9)):
        raise ValueError("PHQ8_ITEMS must contain exactly item IDs 1-8.")

    for item_id, item in PHQ8_ITEMS.items():
        missing = required_keys - set(item.keys())
        if missing:
            raise ValueError(f"Item {item_id} is missing required keys: {missing}")

        if item["item_id"] != item_id:
            raise ValueError(f"Item key {item_id} does not match item_id={item['item_id']}.")

        for key in required_keys:
            if item[key] is None or str(item[key]).strip() == "":
                raise ValueError(f"Item {item_id} has empty value for key: {key}")

    return True


if __name__ == "__main__":
    validate_phq8_items()
    print("PHQ8_ITEMS is valid.")

**3. build_item_dataset**

In [ ]:
%%writefile src/data/build_item_dataset.py
"""
Build an item-level PHQ-8 dataset.

Input:
- Raw transcript CSV files
- Detailed_PHQ8_Labels.csv
- PHQ8_ITEMS mapping

Output:
- data/processed/phq8_item_dataset.csv

Each participant becomes up to 8 rows:
participant_id + item_id -> label
"""

from pathlib import Path
import argparse
import sys
import re

import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parents[2]
sys.path.append(str(PROJECT_ROOT))

from src.data.phq8_items import PHQ8_ITEMS, validate_phq8_items


PARTICIPANT_COL = "Participant_ID"

TEXT_COLUMN_CANDIDATES = [
    "Text",
    "text",
    "Value",
    "value",
    "Utterance",
    "utterance",
    "transcript_text",
]


REQUIRED_OUTPUT_COLUMNS = [
    "participant_id",
    "item_id",
    "item_name",
    "item_text",
    "label",
    "transcript_text",
]


def normalize_participant_id(value):
    """
    Normalize participant IDs so that numeric IDs like 300.0 become '300'.
    """
    if pd.isna(value):
        return None

    text = str(value).strip()
    text = re.sub(r"\.0$", "", text)

    return text


def find_labels_file(labels_dir: Path) -> Path:
    candidates = sorted(labels_dir.rglob("Detailed_PHQ8_Labels.csv"))

    if not candidates:
        raise FileNotFoundError(
            f"Detailed_PHQ8_Labels.csv was not found under: {labels_dir}"
        )

    return candidates[0]


def detect_text_column(transcript_df: pd.DataFrame) -> str:
    for col in TEXT_COLUMN_CANDIDATES:
        if col in transcript_df.columns:
            return col

    raise ValueError(
        "Could not detect transcript text column. "
        f"Available columns: {list(transcript_df.columns)}. "
        f"Expected one of: {TEXT_COLUMN_CANDIDATES}"
    )


def read_transcript_text(transcript_path: Path) -> str:
    df = pd.read_csv(transcript_path)

    if df.empty:
        raise ValueError(f"Transcript file is empty: {transcript_path}")

    text_col = detect_text_column(df)

    texts = (
        df[text_col]
        .dropna()
        .astype(str)
        .map(str.strip)
    )

    texts = texts[texts != ""]

    transcript_text = "\n".join(texts.tolist()).strip()

    if transcript_text == "":
        raise ValueError(f"Transcript text is empty after cleaning: {transcript_path}")

    return transcript_text


def validate_label_value(value, participant_id, item_id):
    numeric_value = pd.to_numeric(value, errors="coerce")

    if pd.isna(numeric_value):
        raise ValueError(
            f"Missing or non-numeric label for participant_id={participant_id}, item_id={item_id}"
        )

    if int(numeric_value) != numeric_value:
        raise ValueError(
            f"Label must be an integer 0-3. "
            f"Got {value} for participant_id={participant_id}, item_id={item_id}"
        )

    label = int(numeric_value)

    if label not in {0, 1, 2, 3}:
        raise ValueError(
            f"Label out of range 0-3. "
            f"Got {label} for participant_id={participant_id}, item_id={item_id}"
        )

    return label


def validate_item_dataset(df: pd.DataFrame):
    missing_cols = [col for col in REQUIRED_OUTPUT_COLUMNS if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Output dataset is missing required columns: {missing_cols}")

    if df.empty:
        raise ValueError("Output dataset is empty.")

    if df["participant_id"].isna().any():
        raise ValueError("participant_id contains missing values.")

    if df["item_id"].isna().any():
        raise ValueError("item_id contains missing values.")

    if not df["item_id"].isin(range(1, 9)).all():
        bad_values = sorted(df.loc[~df["item_id"].isin(range(1, 9)), "item_id"].unique())
        raise ValueError(f"item_id values must be in range 1-8. Bad values: {bad_values}")

    if df["label"].isna().any():
        raise ValueError("label contains missing values.")

    if not df["label"].isin([0, 1, 2, 3]).all():
        bad_values = sorted(df.loc[~df["label"].isin([0, 1, 2, 3]), "label"].unique())
        raise ValueError(f"label values must be 0, 1, 2, 3. Bad values: {bad_values}")

    for col in ["item_name", "item_text", "transcript_text"]:
        empty_mask = df[col].isna() | (df[col].astype(str).str.strip() == "")
        if empty_mask.any():
            raise ValueError(f"{col} contains empty values.")

    rows_per_participant = df.groupby("participant_id").size()
    too_many = rows_per_participant[rows_per_participant > 8]
    if not too_many.empty:
        raise ValueError(
            "Some participants have more than 8 rows: "
            f"{too_many.to_dict()}"
        )

    duplicated_items = df.duplicated(subset=["participant_id", "item_id"])
    if duplicated_items.any():
        examples = df.loc[duplicated_items, ["participant_id", "item_id"]].head()
        raise ValueError(
            "Duplicate participant_id + item_id rows detected. "
            f"Examples:\n{examples}"
        )

    return True


def build_item_dataset(project_dir: Path, labels_path: Path, transcripts_dir: Path, output_path: Path):
    validate_phq8_items()

    if labels_path is None:
        labels_path = find_labels_file(project_dir / "data" / "raw" / "edaic" / "labels")

    if transcripts_dir is None:
        transcripts_dir = project_dir / "data" / "raw" / "edaic" / "transcripts"

    if output_path is None:
        output_path = project_dir / "data" / "processed" / "phq8_item_dataset.csv"

    labels_df = pd.read_csv(labels_path)

    if PARTICIPANT_COL not in labels_df.columns:
        raise ValueError(
            f"Participant column was not found: {PARTICIPANT_COL}. "
            f"Available columns: {list(labels_df.columns)}"
        )

    phq8_columns = [item["column"] for item in PHQ8_ITEMS.values()]
    missing_phq_cols = [col for col in phq8_columns if col not in labels_df.columns]
    if missing_phq_cols:
        raise ValueError(f"Missing PHQ-8 item columns in labels file: {missing_phq_cols}")

    labels_df["_participant_id"] = labels_df[PARTICIPANT_COL].map(normalize_participant_id)

    if labels_df["_participant_id"].isna().any():
        raise ValueError("Some labels rows have missing participant IDs.")

    duplicated_label_ids = labels_df["_participant_id"].duplicated()
    if duplicated_label_ids.any():
        duplicate_ids = labels_df.loc[duplicated_label_ids, "_participant_id"].tolist()
        raise ValueError(f"Duplicate participant IDs in labels file: {duplicate_ids[:20]}")

    labels_by_id = labels_df.set_index("_participant_id")

    transcript_files = sorted(transcripts_dir.glob("*_Transcript.csv"))
    if not transcript_files:
        raise FileNotFoundError(f"No transcript files found in: {transcripts_dir}")

    rows = []
    skipped_no_labels = []

    for transcript_path in transcript_files:
        participant_id = transcript_path.name.replace("_Transcript.csv", "")
        participant_id = normalize_participant_id(participant_id)

        if participant_id not in labels_by_id.index:
            skipped_no_labels.append(participant_id)
            continue

        transcript_text = read_transcript_text(transcript_path)
        label_row = labels_by_id.loc[participant_id]

        for item_id, item in sorted(PHQ8_ITEMS.items()):
            label = validate_label_value(
                value=label_row[item["column"]],
                participant_id=participant_id,
                item_id=item_id,
            )

            rows.append(
                {
                    "participant_id": participant_id,
                    "item_id": item_id,
                    "item_name": item["symptom_name"],
                    "item_text": item["item_text"],
                    "label": label,
                    "transcript_text": transcript_text,
                }
            )

    item_df = pd.DataFrame(rows, columns=REQUIRED_OUTPUT_COLUMNS)
    validate_item_dataset(item_df)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    item_df.to_csv(output_path, index=False)

    n_participants = item_df["participant_id"].nunique()

    print("Item-level dataset created successfully.")
    print(f"Output path: {output_path}")
    print(f"Rows: {len(item_df)}")
    print(f"Participants: {n_participants}")
    print(f"Skipped transcript files without labels: {len(skipped_no_labels)}")

    return item_df


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--project_dir",
        type=str,
        default=str(PROJECT_ROOT),
        help="Project root directory.",
    )

    parser.add_argument(
        "--labels_path",
        type=str,
        default=None,
        help="Optional path to Detailed_PHQ8_Labels.csv.",
    )

    parser.add_argument(
        "--transcripts_dir",
        type=str,
        default=None,
        help="Optional path to transcript CSV directory.",
    )

    parser.add_argument(
        "--output_path",
        type=str,
        default=None,
        help="Optional output path for phq8_item_dataset.csv.",
    )

    return parser.parse_args()


def main():
    args = parse_args()

    project_dir = Path(args.project_dir)

    labels_path = Path(args.labels_path) if args.labels_path else None
    transcripts_dir = Path(args.transcripts_dir) if args.transcripts_dir else None
    output_path = Path(args.output_path) if args.output_path else None

    build_item_dataset(
        project_dir=project_dir,
        labels_path=labels_path,
        transcripts_dir=transcripts_dir,
        output_path=output_path,
    )


if __name__ == "__main__":
    main()

**running - build_item_dataset**

In [ ]:
!python src/data/build_item_dataset.py --project_dir "$PROJECT_DIR"

In [ ]:
import pandas as pd

path = PROJECT_DIR / "data" / "processed" / "phq8_item_dataset.csv"
df = pd.read_csv(path)

print(df.shape)
display(df.head())
print(df["label"].value_counts().sort_index())
print(df.groupby("participant_id").size().describe())

**4. dataset_loader**

In [ ]:
%%writefile src/data/dataset_loader.py
"""
Unified loader for the processed PHQ-8 item-level dataset.

Person B should use this loader instead of reading the raw data directly.
"""

from pathlib import Path
import argparse

import pandas as pd


REQUIRED_COLUMNS = [
    "participant_id",
    "item_id",
    "item_name",
    "item_text",
    "label",
    "transcript_text",
]


def load_item_dataset(path):
    """
    Load and validate the processed item-level PHQ-8 dataset.

    Parameters
    ----------
    path : str or pathlib.Path
        Path to phq8_item_dataset.csv.

    Returns
    -------
    pandas.DataFrame
        Validated item-level dataset.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Dataset file does not exist: {path}")

    df = pd.read_csv(path)

    missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if df["participant_id"].isna().any():
        raise ValueError("participant_id contains missing values.")

    df["participant_id"] = df["participant_id"].astype(str)

    item_id_numeric = pd.to_numeric(df["item_id"], errors="coerce")
    if item_id_numeric.isna().any():
        raise ValueError("item_id contains non-numeric or missing values.")

    df["item_id"] = item_id_numeric.astype(int)

    if not df["item_id"].isin(range(1, 9)).all():
        bad_values = sorted(df.loc[~df["item_id"].isin(range(1, 9)), "item_id"].unique())
        raise ValueError(f"item_id must be in range 1-8. Bad values: {bad_values}")

    label_numeric = pd.to_numeric(df["label"], errors="coerce")
    if label_numeric.isna().any():
        raise ValueError("label contains non-numeric or missing values.")

    df["label"] = label_numeric.astype(int)

    if not df["label"].isin([0, 1, 2, 3]).all():
        bad_values = sorted(df.loc[~df["label"].isin([0, 1, 2, 3]), "label"].unique())
        raise ValueError(f"label must be 0, 1, 2, or 3. Bad values: {bad_values}")

    for col in ["item_name", "item_text", "transcript_text"]:
        empty_mask = df[col].isna() | (df[col].astype(str).str.strip() == "")
        if empty_mask.any():
            raise ValueError(f"{col} contains missing or empty values.")

    if "split" in df.columns:
        valid_splits = {"train", "validation", "test"}
        invalid_splits = set(df["split"].dropna().unique()) - valid_splits
        if invalid_splits:
            raise ValueError(f"Invalid split values detected: {invalid_splits}")

    return df


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("path", type=str, help="Path to phq8_item_dataset.csv")
    args = parser.parse_args()

    df = load_item_dataset(args.path)
    print("Dataset loaded successfully.")
    print(f"Rows: {len(df)}")
    print(f"Participants: {df['participant_id'].nunique()}")
    print(f"Columns: {list(df.columns)}")


if __name__ == "__main__":
    main()

**running dataset_loader**

For future use:

      from src.data.dataset_loader import load_item_dataset

      df = load_item_dataset("data/processed/phq8_item_dataset.csv")

      display(df.head())

In [ ]:
!python src/data/dataset_loader.py data/processed/phq8_item_dataset.csv

**create_splits.py**

In [ ]:
%%writefile src/data/create_splits.py
"""
Create participant-level train/validation/test splits.

Critical rule:
The same participant_id must appear in one split only.
"""

from pathlib import Path
import argparse
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path(__file__).resolve().parents[2]
sys.path.append(str(PROJECT_ROOT))

from src.data.dataset_loader import load_item_dataset


VALID_SPLITS = {"train", "validation", "test"}


def create_splits(
    df: pd.DataFrame,
    train_frac: float = 0.70,
    validation_frac: float = 0.15,
    seed: int = 42,
):
    if train_frac <= 0 or validation_frac <= 0:
        raise ValueError("train_frac and validation_frac must be positive.")

    if train_frac + validation_frac >= 1:
        raise ValueError("train_frac + validation_frac must be less than 1.")

    participants = sorted(df["participant_id"].astype(str).unique())

    if len(participants) < 3:
        raise ValueError("At least 3 participants are required to create train/validation/test splits.")

    rng = np.random.default_rng(seed)
    participants = np.array(participants)
    rng.shuffle(participants)

    n = len(participants)

    n_train = int(round(n * train_frac))
    n_validation = int(round(n * validation_frac))

    # Ensure all three splits are non-empty
    if n_train < 1:
        n_train = 1
    if n_validation < 1:
        n_validation = 1
    if n_train + n_validation >= n:
        n_train = n - 2
        n_validation = 1

    train_ids = participants[:n_train]
    validation_ids = participants[n_train:n_train + n_validation]
    test_ids = participants[n_train + n_validation:]

    split_map = {}

    for pid in train_ids:
        split_map[pid] = "train"

    for pid in validation_ids:
        split_map[pid] = "validation"

    for pid in test_ids:
        split_map[pid] = "test"

    df = df.copy()
    df["participant_id"] = df["participant_id"].astype(str)
    df["split"] = df["participant_id"].map(split_map)

    if df["split"].isna().any():
        missing = df.loc[df["split"].isna(), "participant_id"].unique().tolist()
        raise ValueError(f"Some participants were not assigned to a split: {missing[:20]}")

    validate_splits(df)

    return df


def validate_splits(df: pd.DataFrame):
    if "split" not in df.columns:
        raise ValueError("Missing split column.")

    invalid_splits = set(df["split"].unique()) - VALID_SPLITS
    if invalid_splits:
        raise ValueError(f"Invalid split values: {invalid_splits}")

    present_splits = set(df["split"].unique())
    missing_splits = VALID_SPLITS - present_splits
    if missing_splits:
        raise ValueError(f"Missing required splits: {missing_splits}")

    participant_split_counts = df.groupby("participant_id")["split"].nunique()
    leaking_participants = participant_split_counts[participant_split_counts > 1]

    if not leaking_participants.empty:
        raise ValueError(
            "Participant-level leakage detected. "
            f"Examples: {leaking_participants.head().to_dict()}"
        )

    return True


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--input_path",
        type=str,
        default="data/processed/phq8_item_dataset.csv",
        help="Path to item-level dataset without splits.",
    )

    parser.add_argument(
        "--output_path",
        type=str,
        default="data/processed/phq8_item_dataset_with_splits.csv",
        help="Output path for item-level dataset with splits.",
    )

    parser.add_argument(
        "--train_frac",
        type=float,
        default=0.70,
        help="Fraction of participants assigned to train.",
    )

    parser.add_argument(
        "--validation_frac",
        type=float,
        default=0.15,
        help="Fraction of participants assigned to validation.",
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Random seed.",
    )

    return parser.parse_args()


def main():
    args = parse_args()

    input_path = Path(args.input_path)
    output_path = Path(args.output_path)

    df = load_item_dataset(input_path)

    df_with_splits = create_splits(
        df=df,
        train_frac=args.train_frac,
        validation_frac=args.validation_frac,
        seed=args.seed,
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    df_with_splits.to_csv(output_path, index=False)

    print("Split dataset created successfully.")
    print(f"Output path: {output_path}")

    print("\nRows per split:")
    print(df_with_splits["split"].value_counts())

    print("\nParticipants per split:")
    print(df_with_splits.groupby("split")["participant_id"].nunique())


if __name__ == "__main__":
    main()

running create_splits.py



In [ ]:
!python src/data/create_splits.py \
  --input_path data/processed/phq8_item_dataset.csv \
  --output_path data/processed/phq8_item_dataset_with_splits.csv

In [ ]:
df_splits = pd.read_csv("data/processed/phq8_item_dataset_with_splits.csv")

display(df_splits.head())
print(df_splits["split"].value_counts())
print(df_splits.groupby("split")["participant_id"].nunique())

# Check that each participant appears in only one split
print(df_splits.groupby("participant_id")["split"].nunique().max())

**check_leakage.py**

In [ ]:
%%writefile src/data/check_leakage.py
"""
Check participant-level data leakage across train/validation/test splits.
"""

from pathlib import Path
import argparse
import sys

import pandas as pd


VALID_SPLITS = {"train", "validation", "test"}


def check_leakage(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"File does not exist: {path}")

    df = pd.read_csv(path)

    required_cols = ["participant_id", "split"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    if df["participant_id"].isna().any():
        raise ValueError("participant_id contains missing values.")

    if df["split"].isna().any():
        raise ValueError("split contains missing values.")

    df["participant_id"] = df["participant_id"].astype(str)
    df["split"] = df["split"].astype(str)

    invalid_splits = set(df["split"].unique()) - VALID_SPLITS
    if invalid_splits:
        raise ValueError(f"Invalid split values detected: {invalid_splits}")

    participant_split_counts = df.groupby("participant_id")["split"].nunique()
    leaking_participants = participant_split_counts[participant_split_counts > 1]

    if not leaking_participants.empty:
        for participant_id in leaking_participants.index:
            splits = sorted(df.loc[df["participant_id"] == participant_id, "split"].unique())
            print(f"Leakage detected for participant_id: {participant_id}; splits={splits}")
        return False

    split_to_participants = {
        split: set(df.loc[df["split"] == split, "participant_id"].unique())
        for split in VALID_SPLITS
    }

    split_pairs = [
        ("train", "validation"),
        ("train", "test"),
        ("validation", "test"),
    ]

    leakage_found = False

    for split_a, split_b in split_pairs:
        overlap = split_to_participants[split_a] & split_to_participants[split_b]

        if overlap:
            leakage_found = True
            for participant_id in sorted(overlap):
                print(
                    f"Leakage detected for participant_id: {participant_id}; "
                    f"appears in both {split_a} and {split_b}"
                )

    if leakage_found:
        return False

    print("No participant-level leakage detected.")
    return True


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--input_path",
        type=str,
        default="data/processed/phq8_item_dataset_with_splits.csv",
        help="Path to dataset with split column.",
    )

    return parser.parse_args()


def main():
    args = parse_args()
    ok = check_leakage(Path(args.input_path))

    if not ok:
        sys.exit(1)


if __name__ == "__main__":
    main()

**running check_leakage**

In [ ]:
!python src/data/check_leakage.py \
  --input_path data/processed/phq8_item_dataset_with_splits.csv

In [ ]:
!python src/data/phq8_items.py

!python src/data/build_item_dataset.py \
  --project_dir "$PROJECT_DIR"

!python src/data/dataset_loader.py \
  data/processed/phq8_item_dataset.csv

!python src/data/create_splits.py \
  --input_path data/processed/phq8_item_dataset.csv \
  --output_path data/processed/phq8_item_dataset_with_splits.csv

!python src/data/check_leakage.py \
  --input_path data/processed/phq8_item_dataset_with_splits.csv

**8. create utterance_bank.json**

In [ ]:
%%writefile src/data/preprocess_transcripts.py
"""
Preprocess E-DAIC transcripts and create an utterance bank.

Main functions:
- clean_transcript(text)
- extract_participant_utterances(transcript)

CLI output:
- data/processed/utterance_bank.json

The script prefers raw transcript CSV files when available, because they may
contain speaker information. If speaker information is unavailable, it falls
back to using all transcript text lines.
"""

from pathlib import Path
import argparse
import json
import re
import sys

import pandas as pd


PROJECT_ROOT = Path(__file__).resolve().parents[2]
sys.path.append(str(PROJECT_ROOT))


TEXT_COLUMN_CANDIDATES = [
    "Text",
    "text",
    "Utterance",
    "utterance",
    "Value",
    "value",
    "transcript_text",
]

SPEAKER_COLUMN_CANDIDATES = [
    "Speaker",
    "speaker",
    "Role",
    "role",
    "speaker_role",
    "Participant",
    "participant",
]


INTERVIEWER_MARKERS = {
    "ellie",
    "interviewer",
    "therapist",
    "clinician",
    "doctor",
    "agent",
    "system",
}

PARTICIPANT_MARKERS = {
    "participant",
    "subject",
    "patient",
    "client",
    "user",
}


def clean_transcript(text):
    """
    Clean a single transcript utterance or text segment.

    Parameters
    ----------
    text : str
        Raw utterance text.

    Returns
    -------
    str
        Cleaned utterance.
    """
    if text is None or pd.isna(text):
        return ""

    text = str(text)

    # Remove common non-verbal markers but keep meaningful text.
    text = re.sub(r"\[.*?\]", " ", text)
    text = re.sub(r"\(.*?\)", " ", text)

    # Normalize whitespace.
    text = text.replace("\r", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def detect_text_column(df):
    for col in TEXT_COLUMN_CANDIDATES:
        if col in df.columns:
            return col

    raise ValueError(
        "Could not detect text column. "
        f"Available columns: {list(df.columns)}. "
        f"Expected one of: {TEXT_COLUMN_CANDIDATES}"
    )


def detect_speaker_column(df):
    for col in SPEAKER_COLUMN_CANDIDATES:
        if col in df.columns:
            return col

    return None


def is_participant_speaker(value):
    """
    Decide whether a speaker value probably belongs to the participant.

    Returns
    -------
    True if participant, False if interviewer, None if unknown.
    """
    if value is None or pd.isna(value):
        return None

    speaker = str(value).strip().lower()

    if speaker == "":
        return None

    if any(marker in speaker for marker in INTERVIEWER_MARKERS):
        return False

    if any(marker in speaker for marker in PARTICIPANT_MARKERS):
        return True

    return None


def extract_participant_utterances(transcript):
    """
    Extract participant utterances from either:
    1. a pandas DataFrame loaded from a raw transcript CSV, or
    2. a string containing transcript text.

    Parameters
    ----------
    transcript : pandas.DataFrame or str
        Raw transcript table or transcript text.

    Returns
    -------
    list[str]
        Cleaned participant utterances.
    """
    utterances = []

    if isinstance(transcript, pd.DataFrame):
        df = transcript.copy()

        if df.empty:
            return []

        text_col = detect_text_column(df)
        speaker_col = detect_speaker_column(df)

        if speaker_col is not None:
            speaker_flags = df[speaker_col].map(is_participant_speaker)

            # If we can identify participant rows, keep only them.
            if (speaker_flags == True).any():
                df = df.loc[speaker_flags == True]

            # Otherwise, remove clear interviewer rows if possible.
            elif (speaker_flags == False).any():
                df = df.loc[speaker_flags != False]

        raw_texts = df[text_col].tolist()

    elif isinstance(transcript, str):
        # transcript_text from the processed dataset is newline-separated.
        raw_texts = transcript.split("\n")

    else:
        raise TypeError(
            "transcript must be either a pandas DataFrame or a string."
        )

    for text in raw_texts:
        cleaned = clean_transcript(text)
        if cleaned != "":
            utterances.append(cleaned)

    return utterances


def normalize_participant_id(value):
    if value is None or pd.isna(value):
        return None

    text = str(value).strip()
    text = re.sub(r"\.0$", "", text)

    return text


def load_participant_ids_from_dataset(dataset_path):
    df = pd.read_csv(dataset_path)

    if "participant_id" not in df.columns:
        raise ValueError("Dataset must contain participant_id column.")

    participant_ids = (
        df["participant_id"]
        .dropna()
        .map(normalize_participant_id)
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    return participant_ids, df


def build_utterance_bank_from_raw_transcripts(participant_ids, transcripts_dir):
    """
    Build utterance bank from raw transcript CSV files.
    """
    utterance_bank = {}
    missing_files = []

    for participant_id in participant_ids:
        transcript_path = transcripts_dir / f"{participant_id}_Transcript.csv"

        if not transcript_path.exists():
            missing_files.append(participant_id)
            continue

        transcript_df = pd.read_csv(transcript_path)
        utterances = extract_participant_utterances(transcript_df)

        if utterances:
            utterance_bank[participant_id] = utterances

    return utterance_bank, missing_files


def build_utterance_bank_from_processed_dataset(df):
    """
    Fallback option: build utterance bank from transcript_text in the processed dataset.
    """
    if "transcript_text" not in df.columns:
        raise ValueError("Dataset must contain transcript_text column.")

    utterance_bank = {}

    unique_df = (
        df[["participant_id", "transcript_text"]]
        .drop_duplicates(subset=["participant_id"])
        .copy()
    )

    for _, row in unique_df.iterrows():
        participant_id = normalize_participant_id(row["participant_id"])
        transcript_text = row["transcript_text"]

        utterances = extract_participant_utterances(transcript_text)

        if utterances:
            utterance_bank[participant_id] = utterances

    return utterance_bank


def validate_utterance_bank(utterance_bank):
    if not isinstance(utterance_bank, dict):
        raise ValueError("utterance_bank must be a dictionary.")

    if len(utterance_bank) == 0:
        raise ValueError("utterance_bank is empty.")

    empty_participants = []

    for participant_id, utterances in utterance_bank.items():
        if not isinstance(participant_id, str) or participant_id.strip() == "":
            raise ValueError("All participant IDs must be non-empty strings.")

        if not isinstance(utterances, list):
            raise ValueError(f"Utterances for participant {participant_id} must be a list.")

        cleaned_utterances = []

        for utt in utterances:
            cleaned = clean_transcript(utt)
            if cleaned != "":
                cleaned_utterances.append(cleaned)

        utterance_bank[participant_id] = cleaned_utterances

        if len(cleaned_utterances) == 0:
            empty_participants.append(participant_id)

    if empty_participants:
        raise ValueError(
            "Some participants have empty utterance lists. "
            f"Examples: {empty_participants[:20]}"
        )

    return True


def save_utterance_bank(utterance_bank, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(utterance_bank, f, ensure_ascii=False, indent=2)

    print("Utterance bank saved successfully.")
    print(f"Output path: {output_path}")
    print(f"Participants: {len(utterance_bank)}")

    total_utterances = sum(len(v) for v in utterance_bank.values())
    print(f"Total utterances: {total_utterances}")

    lengths = [len(v) for v in utterance_bank.values()]
    print(f"Min utterances per participant: {min(lengths)}")
    print(f"Max utterances per participant: {max(lengths)}")


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--dataset_path",
        type=str,
        default="data/processed/phq8_item_dataset_with_splits.csv",
        help="Path to processed item-level dataset with splits.",
    )

    parser.add_argument(
        "--transcripts_dir",
        type=str,
        default="data/raw/edaic/transcripts",
        help="Path to raw transcript CSV files.",
    )

    parser.add_argument(
        "--output_path",
        type=str,
        default="data/processed/utterance_bank.json",
        help="Output path for utterance_bank.json.",
    )

    parser.add_argument(
        "--fallback_to_processed",
        action="store_true",
        help="Use transcript_text from processed dataset if raw transcript files are unavailable.",
    )

    return parser.parse_args()


def main():
    args = parse_args()

    dataset_path = Path(args.dataset_path)
    transcripts_dir = Path(args.transcripts_dir)
    output_path = Path(args.output_path)

    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset file does not exist: {dataset_path}")

    participant_ids, df = load_participant_ids_from_dataset(dataset_path)

    if not transcripts_dir.exists():
        if args.fallback_to_processed:
            print("Raw transcripts directory not found. Falling back to processed transcript_text.")
            utterance_bank = build_utterance_bank_from_processed_dataset(df)
        else:
            raise FileNotFoundError(
                f"Raw transcripts directory does not exist: {transcripts_dir}. "
                "Use --fallback_to_processed to build from transcript_text instead."
            )
    else:
        utterance_bank, missing_files = build_utterance_bank_from_raw_transcripts(
            participant_ids=participant_ids,
            transcripts_dir=transcripts_dir,
        )

        print(f"Missing raw transcript files: {len(missing_files)}")

        if len(utterance_bank) == 0 and args.fallback_to_processed:
            print("No utterances extracted from raw transcripts. Falling back to processed transcript_text.")
            utterance_bank = build_utterance_bank_from_processed_dataset(df)

    validate_utterance_bank(utterance_bank)
    save_utterance_bank(utterance_bank, output_path)


if __name__ == "__main__":
    main()

In [ ]:
!python src/data/preprocess_transcripts.py \
  --dataset_path data/processed/phq8_item_dataset_with_splits.csv \
  --transcripts_dir data/raw/edaic/transcripts \
  --output_path data/processed/utterance_bank.json \
  --fallback_to_processed

In [ ]:
import json
from pathlib import Path

utterance_bank_path = Path("data/processed/utterance_bank.json")

with open(utterance_bank_path, "r", encoding="utf-8") as f:
    utterance_bank = json.load(f)

print("Participants:", len(utterance_bank))

first_pid = next(iter(utterance_bank))
print("Example participant:", first_pid)
print("Number of utterances:", len(utterance_bank[first_pid]))

utterance_bank[first_pid][:5]

In [ ]:
empty = {
    pid: utterances
    for pid, utterances in utterance_bank.items()
    if len(utterances) == 0
}

print("Participants with empty utterance lists:", len(empty))

In [ ]:
!python src/data/preprocess_transcripts.py \
  --dataset_path data/processed/phq8_item_dataset_with_splits.csv \
  --transcripts_dir data/raw/edaic/transcripts \
  --output_path data/processed/utterance_bank.json \
  --fallback_to_processed